In [ ]:
"""
SV callset preprocessing notebook
"""

import os

workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2"
os.chdir(workdir)
os.getcwd()

import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from pysam import VariantFile

import svensemble.vcfparse as vcfp
import svensemble._caller_params as caller_params
import svensemble.breakends as bnds
import svensemble.breakpoints as bps
import svensemble.events as evts
import svensemble.caller_specific as caller_specific
import svensemble.utils as sv_utils
import svensemble.pipeline as sv_pipeline

importlib.reload(vcfp)
importlib.reload(caller_params)
importlib.reload(bnds)
importlib.reload(bps)
importlib.reload(evts)
importlib.reload(caller_specific)
importlib.reload(sv_utils)
importlib.reload(sv_pipeline)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

preprocessed_savedir = Path('saved/preprocessed/')
preprocessed_savedir.mkdir(parents=True, exist_ok=True)

ref_path = Path('/home/users/allstaff/shikov.v/projects/Papenfuss_SVEnsemble2/data/hg38/downloads/hg38.no_decoys.fa')

samples = {}
for sample_name in ['CHM13', 'NA12878', 'NA12878_30x']:
    samples.update(sv_utils.create_config_dict(sample_name, ref_path=ref_path, is_ground_truth=False))
samples.update(sv_utils.create_config_dict('gnomad', is_ground_truth=True))

samples

# callsets = {
#     'gnomad': {
#         'source': 'gnomad',
#         'path': Path('data/gnomad.v4.1.sv.sites.vcf.gz'),
#         'params': caller_params.gnomad_params
#     },
#     'gridss': {
#         'source': 'gridss',
#         'path': Path('data/test_gridss_output.NA12878.vcf'),
#         'params': caller_params.gridss_params
#     },
#     'manta_candidate': {
#         'source': 'manta_candidate',
#         'path': Path('data/candidateSV.vcf.gz'),
#         'params': caller_params.manta_candidate_params
#     },
#     'manta_diploid': {
#         'source': 'manta_diploid',
#         'path': Path('data/diploidSV.vcf.gz'),
#         'params': caller_params.manta_diploid_params
#     }
# }

to_process_list = ['delly']

In [ ]:
delly_vcf = VariantFile(sample_config['NA12878_30x']['callsets']['delly']['path'])
delly_df, delly_meta = vcfp.parse_vcf_to_df(vcf_file=delly_vcf, caller_params=sample_config['NA12878_30x']['callsets']['delly']['params'])

delly_bnd_only_df = delly_df[delly_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
delly_evt_only_df = delly_df[~delly_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
delly_bnd_only_df = caller_specific.delly_preprocess_bnd_only(delly_bnd_only_df, caller_params=sample_config['NA12878_30x']['callsets']['delly']['params'])
delly_evt_only_df = caller_specific.delly_preprocess_evts_only(delly_evt_only_df, caller_params=sample_config['NA12878_30x']['callsets']['delly']['params'])
manta_evt_bnd_df = evts.event_to_bnd(manta_evt_only_df)
manta_diploid_df = pd.concat([manta_bnd_only_df, manta_evt_bnd_df], axis=0)
manta_diploid_bp_df = bnds.bnd_to_bp(manta_diploid_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])

In [ ]:
delly_df[delly_df['INFO_SVTYPE'] == 'BND']

In [ ]:
for filter_field, filter_field_content in test_delly_vcf.header.filters.items():
    print(f'FILTER: {filter_field}: {filter_field_content.description} ({filter_field_content.type}, {filter_field_content.number})')

for info_field, info_field_content in test_delly_vcf.header.info.items():
    print(f'INFO: {info_field}: {info_field_content.description} ({info_field_content.type}, {info_field_content.number})')

for format_field, format_field_content in test_delly_vcf.header.formats.items():
    print(f'FORMAT: {format_field}: {format_field_content.description} ({format_field_content.type}, {format_field_content.number})')

In [ ]:
mandatory = ['']

In [ ]:
samples['gnomad']

In [ ]:
for sample, sample_config in samples.items():
    if sample_config['is_reference']:
        # Check if the reference sample contains only 1 "callset"
        if len(sample_config['callsets']) > 1:
            raise ValueError(f'Reference sample {sample} contains multiple callsets: {sample_config["callsets"].keys()}')
        # Preprocess the reference sample
        if sample_config['callsets'][sample_config['callsets'].keys()[0]]['source'] == 'gnomad':
            gnomad_vcf = VariantFile(sample_config['callsets'][sample]['path'])
            gnomad_df, gnomad_meta = vcfp.parse_vcf_to_df(vcf_file=gnomad_vcf, caller_params=sample_config['callsets'][sample]['params'])
            gnomad_df = caller_specific.gnomad_preprocess(gnomad_df, caller_params=sample_config['callsets'][sample]['params'], verbose=True)
            gnomad_bnd_df = caller_specific.gnomad_event_to_bnd(gnomad_df)
            gnomad_bp_df = bnds.bnd_to_bp(gnomad_bnd_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
            if not (sample_config['sample_dir'] / 'svensemble2').exists():
                (sample_config['sample_dir'] / 'svensemble2').mkdir(parents=True, exist_ok=True)
            gnomad_bp_df.to_csv(sample_config['sample_dir'] / 'svensemble2' / 'gnomad_bps.csv')
            del gnomad_df, gnomad_meta, gnomad_bnd_df, gnomad_bp_df
        else:
            raise ValueError(f'Unknown reference sample: {sample}')
        continue

    if 'gridss' in sample_config['callsets'].keys():
        gridss_vcf = VariantFile(sample_config['callsets']['gridss']['path'])
        gridss_df, gridss_meta = vcfp.parse_vcf_to_df(vcf_file=gridss_vcf, caller_params=sample_config['callsets']['gridss']['params'])
        gridss_df = caller_specific.gridss_preprocess(gridss_df, caller_params=sample_config['callsets']['gridss']['params'], verbose=True)
        gridss_bp_df = bnds.bnd_to_bp(gridss_df)
        if not (sample_config['sample_dir'] / 'sv_calling' / 'gridss' / 'svensemble2').exists():
            (sample_config['sample_dir'] / 'sv_calling' / 'gridss' / 'svensemble2').mkdir(parents=True, exist_ok=True)
        gridss_bp_df.to_csv(sample_config['sample_dir'] / 'sv_calling' / 'gridss' / 'svensemble2' / 'gridss_bps.csv')
        del gridss_df, gridss_meta, gridss_bp_df
    if 'manta_diploid' in to_process_list:
        manta_diploid_vcf = VariantFile(sample_config['callsets']['manta_diploid']['path'])
        manta_diploid_df, manta_diploid_meta = vcfp.parse_vcf_to_df(vcf_file=manta_diploid_vcf, caller_params=sample_config['callsets']['manta_diploid']['params'])
        manta_bnd_only_df = manta_diploid_df[manta_diploid_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
        manta_evt_only_df = manta_diploid_df[~manta_diploid_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
        manta_bnd_only_df = caller_specific.manta_preprocess_bnd_only(manta_bnd_only_df, caller_params=sample_config['callsets']['manta_diploid']['params'])
        manta_evt_only_df = caller_specific.manta_preprocess_evts_only(manta_evt_only_df, caller_params=sample_config['callsets']['manta_diploid']['params'])
        manta_evt_bnd_df = evts.event_to_bnd(manta_evt_only_df)
        manta_diploid_df = pd.concat([manta_bnd_only_df, manta_evt_bnd_df], axis=0)
        manta_diploid_bp_df = bnds.bnd_to_bp(manta_diploid_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
        if not (sample_config['sample_dir'] / 'sv_calling' / 'manta' / 'svensemble2').exists():
            (sample_config['sample_dir'] / 'sv_calling' / 'manta' / 'svensemble2').mkdir(parents=True, exist_ok=True)
        manta_diploid_bp_df.to_csv(sample_config['sample_dir'] / 'sv_calling' / 'manta' / 'svensemble2' / 'manta_diploid_bps.csv')
        del manta_diploid_df, manta_diploid_meta, manta_bnd_only_df, manta_evt_only_df, manta_evt_bnd_df, manta_diploid_bp_df
    if 'manta_candidate' in sample_config['callsets'].keys():
        manta_candidate_vcf = VariantFile(sample_config['callsets']['manta_candidate']['path'])
        manta_candidate_df, manta_candidate_meta = vcfp.parse_vcf_to_df(vcf_file=manta_candidate_vcf, caller_params=sample_config['callsets']['manta_candidate']['params'])
        manta_candidate_bnd_only_df = manta_candidate_df[manta_candidate_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
        manta_candidate_evt_only_df = manta_candidate_df[~manta_candidate_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
        manta_candidate_bnd_only_df = caller_specific.manta_preprocess_bnd_only(manta_candidate_bnd_only_df, caller_params=sample_config['callsets']['manta_candidate']['params'])
        manta_candidate_evt_only_df = caller_specific.manta_preprocess_evts_only(manta_candidate_evt_only_df, caller_params=sample_config['callsets']['manta_candidate']['params'])
        manta_candidate_evt_bnd_df = evts.event_to_bnd(manta_candidate_evt_only_df)
        manta_candidate_df = pd.concat([manta_candidate_bnd_only_df, manta_candidate_evt_bnd_df], axis=0)
        manta_candidate_bp_df = bnds.bnd_to_bp(manta_candidate_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
        if not (sample_config['sample_dir'] / 'sv_calling' / 'manta' / 'svensemble2').exists():
            (sample_config['sample_dir'] / 'sv_calling' / 'manta' / 'svensemble2').mkdir(parents=True, exist_ok=True)
        manta_candidate_bp_df.to_csv(sample_config['sample_dir'] / 'sv_calling' / 'manta' / 'svensemble2' / 'manta_candidate_bps.csv')
        del manta_candidate_df, manta_candidate_meta, manta_candidate_bnd_only_df, manta_candidate_evt_only_df, manta_candidate_evt_bnd_df, manta_candidate_bp_df


In [ ]:
if 'gridss' in to_process_list:
    gridss_vcf = VariantFile(callsets['gridss']['path'])
    gridss_df, gridss_meta = vcfp.parse_vcf_to_df(vcf_file=gridss_vcf, caller_params=callsets['gridss']['params'])
    gridss_df = caller_specific.gridss_preprocess(gridss_df, caller_params=callsets['gridss']['params'], verbose=True)
    gridss_bp_df = bnds.bnd_to_bp(gridss_df)
    gridss_bp_df.to_csv(preprocessed_savedir / 'gridss_bps.csv')
    del gridss_df, gridss_meta, gridss_bp_df


In [ ]:
if 'gnomad' in to_process_list:
    gnomad_vcf = VariantFile(callsets['gnomad']['path'])
    gnomad_df, gnomad_meta = vcfp.parse_vcf_to_df(vcf_file=gnomad_vcf, caller_params=callsets['gnomad']['params'])
    gnomad_df = caller_specific.gnomad_preprocess(gnomad_df, caller_params=callsets['gnomad']['params'], verbose=True)
    gnomad_bnd_df = caller_specific.gnomad_event_to_bnd(gnomad_df)
    gnomad_bp_df = bnds.bnd_to_bp(gnomad_bnd_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
    gnomad_bp_df.to_csv(preprocessed_savedir / 'gnomad_bps.csv')
    del gnomad_df, gnomad_meta, gnomad_bnd_df, gnomad_bp_df


In [ ]:
if 'manta_diploid' in to_process_list:
    manta_diploid_vcf = VariantFile(callsets['manta_diploid']['path'])
    manta_diploid_df, manta_diploid_meta = vcfp.parse_vcf_to_df(vcf_file=manta_diploid_vcf, caller_params=callsets['manta_diploid']['params'])
    manta_bnd_only_df = manta_diploid_df[manta_diploid_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
    manta_evt_only_df = manta_diploid_df[~manta_diploid_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
    manta_bnd_only_df = caller_specific.manta_preprocess_bnd_only(manta_bnd_only_df, caller_params=callsets['manta_diploid']['params'])
    manta_evt_only_df = caller_specific.manta_preprocess_evts_only(manta_evt_only_df, caller_params=callsets['manta_diploid']['params'])
    manta_evt_bnd_df = evts.event_to_bnd(manta_evt_only_df)
    manta_diploid_df = pd.concat([manta_bnd_only_df, manta_evt_bnd_df], axis=0)
    manta_diploid_bp_df = bnds.bnd_to_bp(manta_diploid_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
    manta_diploid_bp_df.to_csv(preprocessed_savedir / 'manta_diploid_bps.csv')
    del manta_diploid_df, manta_diploid_meta, manta_bnd_only_df, manta_evt_only_df, manta_evt_bnd_df, manta_diploid_bp_df

In [ ]:
manta_diploid_df

In [ ]:
manta_diploid_bp_df

In [ ]:
manta_evt_only_df['CHROM'].dtype

In [ ]:
if 'manta_candidate' in to_process_list:
    manta_candidate_vcf = VariantFile(callsets['manta_candidate']['path'])
    manta_candidate_df, manta_candidate_meta = vcfp.parse_vcf_to_df(vcf_file=manta_candidate_vcf, field_info=callsets['manta_candidate']['params'])
    manta_candidate_bnd_only_df = manta_candidate_df[manta_candidate_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
    manta_candidate_evt_only_df = manta_candidate_df[~manta_candidate_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
    manta_candidate_bnd_only_df = caller_specific.manta_preprocess_bnd_only(manta_candidate_bnd_only_df)
    manta_candidate_evt_only_df = caller_specific.manta_preprocess_evts_only(manta_candidate_evt_only_df)
    manta_candidate_evt_bnd_df = evts.event_to_bnd(manta_candidate_evt_only_df)
    manta_candidate_df = pd.concat([manta_candidate_bnd_only_df, manta_candidate_evt_bnd_df], axis=0)
    manta_candidate_bp_df = bnds.bnd_to_bp(manta_candidate_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
    manta_candidate_bp_df.to_csv(preprocessed_savedir / 'manta_candidate_bps.csv')
    del manta_candidate_df, manta_candidate_meta, manta_candidate_bnd_only_df, manta_candidate_evt_only_df, manta_candidate_evt_bnd_df, manta_candidate_bp_df